# MELBA 0.1.0: business sales regression

This notebook demonstrates MELBA on a **synthetic** dataset. The generated values do not come from a company or a real market. MELBA uses a quadratic feature map with Huber loss and L2 regularization. The notebook reports metrics on a held-out split.

The notebook is designed for use from a MELBA checkout or as a standalone Colab notebook. When no checkout is present, the bundled source files below are written into the Colab runtime and imported directly. The data, training, and evaluation cells were run locally; a live Colab session and the optional plotting cell were not tested here.


In [ ]:
%pip install -q numpy pandas matplotlib


In [ ]:
from pathlib import Path
import sys
SOURCE_FILES = {'C:/Users/Lenovo/Documents/MELBA/melba/__init__.py': '"""MELBA: a small business-data regression library."""\n\nfrom .model import MELBA\n\n__version__ = "0.1.0"\n\n__all__ = ["MELBA", "__version__"]\n', 'C:/Users/Lenovo/Documents/MELBA/melba/core/__init__.py': '"""Learning, optimization, and input validation primitives."""\n\n', 'C:/Users/Lenovo/Documents/MELBA/melba/core/learning.py': '"""Huber objective and derivative used by MELBA\'s trainer."""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef huber_loss(residuals: np.ndarray, delta: float) -> np.ndarray:\n    """Compute elementwise Huber loss for residuals and threshold ``delta``."""\n    absolute = np.abs(residuals)\n    return np.where(\n        absolute <= delta,\n        0.5 * residuals**2,\n        delta * (absolute - 0.5 * delta),\n    )\n\n\ndef huber_gradient(residuals: np.ndarray, delta: float) -> np.ndarray:\n    """Compute the derivative of Huber loss with respect to each residual."""\n    return np.clip(residuals, -delta, delta)\n\n', 'C:/Users/Lenovo/Documents/MELBA/melba/core/optimization.py': '"""Full-batch gradient descent for the regularized Huber objective."""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .learning import huber_gradient, huber_loss\n\n\ndef optimize_huber(\n    design: np.ndarray,\n    target: np.ndarray,\n    *,\n    learning_rate: float,\n    epochs: int,\n    tolerance: float,\n    l2_penalty: float,\n    huber_delta: float,\n    random_state: int | None,\n) -> tuple[np.ndarray, list[float], bool]:\n    """Fit weights using full-batch gradient descent.\n\n    ``design`` excludes the intercept. The intercept is prepended here and is\n    left out of the L2 penalty. The Frobenius norm provides a safe upper bound\n    for the smooth part\'s gradient Lipschitz constant.\n    """\n    sample_count, feature_count = design.shape\n    augmented = np.column_stack((np.ones(sample_count), design))\n    regularized = np.ones(feature_count + 1)\n    regularized[0] = 0.0\n\n    generator = np.random.default_rng(random_state)\n    weights = np.zeros(feature_count + 1)\n    weights[1:] = generator.normal(0.0, 0.01, size=feature_count)\n\n    lipschitz_bound = float(np.sum(augmented * augmented) / sample_count + l2_penalty)\n    step_size = learning_rate / max(lipschitz_bound, np.finfo(float).eps)\n\n    def objective(current: np.ndarray) -> float:\n        residuals = augmented @ current - target\n        data_loss = float(np.mean(huber_loss(residuals, huber_delta)))\n        penalty = 0.5 * l2_penalty * float(np.sum(current[1:] ** 2))\n        return data_loss + penalty\n\n    history = [objective(weights)]\n    converged = False\n    for _ in range(epochs):\n        residuals = augmented @ weights - target\n        gradient = augmented.T @ huber_gradient(residuals, huber_delta) / sample_count\n        gradient[1:] += l2_penalty * weights[1:]\n        weights -= step_size * gradient\n\n        current_loss = objective(weights)\n        history.append(current_loss)\n        if abs(history[-2] - current_loss) <= tolerance:\n            converged = True\n            break\n\n    return weights, history, converged\n\n', 'C:/Users/Lenovo/Documents/MELBA/melba/core/validation.py': '"""Input checks shared by the model and evaluation metrics."""\n\nfrom __future__ import annotations\n\nfrom numbers import Integral, Real\n\nimport numpy as np\n\n\ndef _real_array(values: object, name: str) -> np.ndarray:\n    try:\n        raw = np.asarray(values)\n    except (TypeError, ValueError) as exc:\n        raise ValueError(f"{name} must be a non-empty numeric array.") from exc\n\n    if raw.dtype.kind not in "biuf":\n        raise ValueError(f"{name} must contain real numeric values.")\n    try:\n        array = np.asarray(raw, dtype=float)\n    except (TypeError, ValueError, OverflowError) as exc:\n        raise ValueError(f"{name} must contain real numeric values.") from exc\n    return array\n\n\ndef as_features(values: object, name: str = "X") -> np.ndarray:\n    """Return features as a finite, two-dimensional float array."""\n    array = _real_array(values, name)\n    if array.ndim == 1:\n        array = array.reshape(-1, 1)\n    elif array.ndim != 2:\n        raise ValueError(f"{name} must be one- or two-dimensional.")\n    if array.shape[0] == 0 or array.shape[1] == 0:\n        raise ValueError(f"{name} must contain at least one sample and one feature.")\n    if not np.isfinite(array).all():\n        raise ValueError(f"{name} must not contain NaN or infinity.")\n    return array\n\n\ndef as_target(values: object, name: str = "y") -> np.ndarray:\n    """Return a finite, one-dimensional target array."""\n    array = _real_array(values, name)\n    if array.ndim == 2 and array.shape[1] == 1:\n        array = array[:, 0]\n    elif array.ndim != 1:\n        raise ValueError(f"{name} must be one-dimensional or a single-column array.")\n    if array.size == 0:\n        raise ValueError(f"{name} must contain at least one sample.")\n    if not np.isfinite(array).all():\n        raise ValueError(f"{name} must not contain NaN or infinity.")\n    return array\n\n\ndef validate_training_data(X: object, y: object) -> tuple[np.ndarray, np.ndarray]:\n    """Validate a feature matrix and its matching target."""\n    features = as_features(X)\n    target = as_target(y)\n    if features.shape[0] != target.size:\n        raise ValueError(\n            "X and y must contain the same number of samples; "\n            f"got {features.shape[0]} and {target.size}."\n        )\n    return features, target\n\n\ndef validate_positive_real(value: object, name: str, *, allow_zero: bool = False) -> float:\n    """Validate and normalize a finite positive real parameter."""\n    if isinstance(value, bool) or not isinstance(value, Real):\n        raise ValueError(f"{name} must be a finite real number.")\n    normalized = float(value)\n    minimum_ok = normalized >= 0 if allow_zero else normalized > 0\n    if not np.isfinite(normalized) or not minimum_ok:\n        qualifier = "non-negative" if allow_zero else "positive"\n        raise ValueError(f"{name} must be finite and {qualifier}.")\n    return normalized\n\n\ndef validate_epochs(value: object) -> int:\n    if isinstance(value, bool) or not isinstance(value, Integral) or value <= 0:\n        raise ValueError("epochs must be a positive integer.")\n    return int(value)\n\n\ndef validate_random_state(value: object) -> int | None:\n    if value is None:\n        return None\n    if isinstance(value, bool) or not isinstance(value, Integral):\n        raise ValueError("random_state must be an integer or None.")\n    return int(value)\n\n', 'C:/Users/Lenovo/Documents/MELBA/melba/evaluation/__init__.py': '"""Regression metrics used by :class:`melba.MELBA`."""\n\nfrom .metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error\n\n__all__ = [\n    "mean_absolute_error",\n    "mean_squared_error",\n    "root_mean_squared_error",\n    "r2_score",\n]\n', 'C:/Users/Lenovo/Documents/MELBA/melba/evaluation/metrics.py': '"""Regression evaluation metrics with explicit constant-target handling."""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom ..core.validation import as_target\n\n\ndef _matching_targets(y_true: object, y_pred: object) -> tuple[np.ndarray, np.ndarray]:\n    actual = as_target(y_true, "y_true")\n    predicted = as_target(y_pred, "y_pred")\n    if actual.size != predicted.size:\n        raise ValueError(\n            "y_true and y_pred must contain the same number of samples; "\n            f"got {actual.size} and {predicted.size}."\n        )\n    return actual, predicted\n\n\ndef mean_absolute_error(y_true: object, y_pred: object) -> float:\n    actual, predicted = _matching_targets(y_true, y_pred)\n    return float(np.mean(np.abs(actual - predicted)))\n\n\ndef mean_squared_error(y_true: object, y_pred: object) -> float:\n    actual, predicted = _matching_targets(y_true, y_pred)\n    return float(np.mean((actual - predicted) ** 2))\n\n\ndef root_mean_squared_error(y_true: object, y_pred: object) -> float:\n    return float(np.sqrt(mean_squared_error(y_true, y_pred)))\n\n\ndef r2_score(y_true: object, y_pred: object) -> float:\n    actual, predicted = _matching_targets(y_true, y_pred)\n    residual_sum = float(np.sum((actual - predicted) ** 2))\n    total_sum = float(np.sum((actual - np.mean(actual)) ** 2))\n    if total_sum == 0.0:\n        return 1.0 if residual_sum == 0.0 else 0.0\n    return float(1.0 - residual_sum / total_sum)\n\n', 'C:/Users/Lenovo/Documents/MELBA/melba/model.py': '"""Public MELBA business-data regression estimator."""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .core.optimization import optimize_huber\nfrom .core.validation import (\n    as_features,\n    as_target,\n    validate_epochs,\n    validate_positive_real,\n    validate_random_state,\n    validate_training_data,\n)\nfrom .evaluation.metrics import (\n    mean_absolute_error,\n    mean_squared_error,\n    r2_score,\n    root_mean_squared_error,\n)\nfrom .utils.preprocessing import BusinessFeatureMap\n\n\nclass MELBA:\n    """Fit a regularized Huber regression model to business data.\n\n    By default, the model adds squared and pairwise feature terms to capture\n    simple curvature and interactions. This is a classical quadratic feature\n    regression formulation, not a claim of a new learning method.\n    """\n\n    def __init__(\n        self,\n        learning_rate: float = 0.5,\n        epochs: int = 1000,\n        tolerance: float = 1e-8,\n        random_state: int | None = 42,\n        l2_penalty: float = 0.01,\n        huber_delta: float = 1.345,\n        quadratic_terms: bool = True,\n    ) -> None:\n        self.learning_rate = validate_positive_real(learning_rate, "learning_rate")\n        if self.learning_rate > 1.0:\n            raise ValueError("learning_rate must be greater than 0 and at most 1.")\n        self.epochs = validate_epochs(epochs)\n        self.tolerance = validate_positive_real(tolerance, "tolerance", allow_zero=True)\n        self.random_state = validate_random_state(random_state)\n        self.l2_penalty = validate_positive_real(l2_penalty, "l2_penalty", allow_zero=True)\n        self.huber_delta = validate_positive_real(huber_delta, "huber_delta")\n        if not isinstance(quadratic_terms, (bool, np.bool_)):\n            raise ValueError("quadratic_terms must be a boolean.")\n        self.quadratic_terms = bool(quadratic_terms)\n\n        self.is_fitted_ = False\n        self.n_features_in_: int | None = None\n        self.feature_map_: BusinessFeatureMap | None = None\n        self.weights_: np.ndarray | None = None\n        self.target_mean_: float | None = None\n        self.target_scale_: float | None = None\n        self.loss_history_: list[float] = []\n        self.loss_: float | None = None\n        self.n_iter_: int = 0\n        self.converged_: bool = False\n\n    def fit(self, X: object, y: object) -> "MELBA":\n        """Validate data and optimize model parameters, then return ``self``."""\n        features, target = validate_training_data(X, y)\n        feature_map = BusinessFeatureMap(self.quadratic_terms)\n        mapped = feature_map.fit_transform(features)\n\n        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):\n            target_mean = float(np.mean(target))\n            target_scale = float(np.std(target))\n        if not np.isfinite(target_mean) or not np.isfinite(target_scale):\n            raise ValueError("y values are too large to normalize safely.")\n        if target_scale <= np.finfo(float).eps:\n            target_scale = 1.0\n        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):\n            normalized_target = (target - target_mean) / target_scale\n        if not np.isfinite(normalized_target).all():\n            raise ValueError("y values are too large to normalize safely.")\n\n        weights, history, converged = optimize_huber(\n            mapped,\n            normalized_target,\n            learning_rate=self.learning_rate,\n            epochs=self.epochs,\n            tolerance=self.tolerance,\n            l2_penalty=self.l2_penalty,\n            huber_delta=self.huber_delta,\n            random_state=self.random_state,\n        )\n\n        self.feature_map_ = feature_map\n        self.weights_ = weights\n        self.target_mean_ = target_mean\n        self.target_scale_ = target_scale\n        self.n_features_in_ = features.shape[1]\n        self.loss_history_ = history\n        self.loss_ = history[-1]\n        self.n_iter_ = len(history) - 1\n        self.converged_ = converged\n        self.is_fitted_ = True\n        return self\n\n    def predict(self, X: object) -> np.ndarray:\n        """Predict target values for one or more feature rows."""\n        if not self.is_fitted_ or self.feature_map_ is None or self.weights_ is None:\n            raise RuntimeError("MELBA must be fitted before calling predict.")\n        features = as_features(X)\n        mapped = self.feature_map_.transform(features)\n        design = np.column_stack((np.ones(features.shape[0]), mapped))\n        assert self.target_mean_ is not None and self.target_scale_ is not None\n        with np.errstate(over="ignore", invalid="ignore"):\n            predictions = design @ self.weights_ * self.target_scale_ + self.target_mean_\n        if not np.isfinite(predictions).all():\n            raise ValueError("Prediction produced nonfinite values; check the input scale.")\n        return predictions\n\n    def evaluate(self, X: object, y: object) -> dict[str, float]:\n        """Return MAE, MSE, RMSE, and R-squared on the supplied data."""\n        actual = as_target(y)\n        predictions = self.predict(X)\n        if actual.size != predictions.size:\n            raise ValueError(\n                "X and y must contain the same number of samples; "\n                f"got {predictions.size} and {actual.size}."\n            )\n        return {\n            "mae": mean_absolute_error(actual, predictions),\n            "mse": mean_squared_error(actual, predictions),\n            "rmse": root_mean_squared_error(actual, predictions),\n            "r2": r2_score(actual, predictions),\n        }\n\n', 'C:/Users/Lenovo/Documents/MELBA/melba/utils/__init__.py': '"""Feature preparation utilities."""\n\nfrom .preprocessing import BusinessFeatureMap\n\n__all__ = ["BusinessFeatureMap"]\n', 'C:/Users/Lenovo/Documents/MELBA/melba/utils/preprocessing.py': '"""Training-fitted standardization and quadratic business feature mapping."""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n\nclass BusinessFeatureMap:\n    """Standardize inputs and optionally add squared and pairwise terms."""\n\n    def __init__(self, quadratic_terms: bool = True) -> None:\n        self.quadratic_terms = quadratic_terms\n        self.n_features_in_: int | None = None\n        self.input_mean_: np.ndarray | None = None\n        self.input_scale_: np.ndarray | None = None\n        self.term_mean_: np.ndarray | None = None\n        self.term_scale_: np.ndarray | None = None\n\n    def _expand(self, standardized: np.ndarray) -> np.ndarray:\n        if not self.quadratic_terms:\n            return standardized\n        count = standardized.shape[1]\n        quadratic = [\n            standardized[:, left] * standardized[:, right]\n            for left in range(count)\n            for right in range(left, count)\n        ]\n        return np.column_stack((standardized, *quadratic))\n\n    def fit_transform(self, features: np.ndarray) -> np.ndarray:\n        self.n_features_in_ = features.shape[1]\n        with np.errstate(over="ignore", invalid="ignore"):\n            self.input_mean_ = np.mean(features, axis=0)\n            input_scale = np.std(features, axis=0)\n        if not np.isfinite(self.input_mean_).all() or not np.isfinite(input_scale).all():\n            raise ValueError("X values are too large to scale safely.")\n        self.input_scale_ = np.where(input_scale > np.finfo(float).eps, input_scale, 1.0)\n\n        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):\n            standardized = (features - self.input_mean_) / self.input_scale_\n            expanded = self._expand(standardized)\n            self.term_mean_ = np.mean(expanded, axis=0)\n            term_scale = np.std(expanded, axis=0)\n        if (\n            not np.isfinite(expanded).all()\n            or not np.isfinite(self.term_mean_).all()\n            or not np.isfinite(term_scale).all()\n        ):\n            raise ValueError("X values are too large to scale safely.")\n        self.term_scale_ = np.where(term_scale > np.finfo(float).eps, term_scale, 1.0)\n        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):\n            mapped = (expanded - self.term_mean_) / self.term_scale_\n        if not np.isfinite(mapped).all():\n            raise ValueError("X values are too large to scale safely.")\n        return mapped\n\n    def transform(self, features: np.ndarray) -> np.ndarray:\n        if self.n_features_in_ is None or self.input_mean_ is None:\n            raise RuntimeError("The feature map must be fitted before transform.")\n        if features.shape[1] != self.n_features_in_:\n            raise ValueError(\n                "X has a different number of features than the data used for training; "\n                f"expected {self.n_features_in_}, got {features.shape[1]}."\n            )\n        assert self.input_scale_ is not None\n        assert self.term_mean_ is not None\n        assert self.term_scale_ is not None\n        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):\n            standardized = (features - self.input_mean_) / self.input_scale_\n            expanded = self._expand(standardized)\n            mapped = (expanded - self.term_mean_) / self.term_scale_\n        if not np.isfinite(mapped).all():\n            raise ValueError("X values are too large to transform safely.")\n        return mapped\n\n'}

project_root = Path.cwd()
if (project_root / 'melba' / '__init__.py').is_file():
    source_root = project_root
else:
    source_root = Path('/content/melba_source')
    for relative_path, source in SOURCE_FILES.items():
        destination = source_root / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(source, encoding='utf-8')
sys.path.insert(0, str(source_root))
from melba import MELBA, __version__
print(f'MELBA {__version__} loaded from {source_root}')


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
sample_count = 320
price = rng.uniform(12.0, 60.0, sample_count)
promotion_spend = rng.uniform(0.0, 80.0, sample_count)
operating_cost = rng.uniform(500.0, 2500.0, sample_count)
sales = (
    280.0 - 2.0 * price + 1.3 * promotion_spend
    - 0.009 * promotion_spend**2 - 0.018 * operating_cost
    + 0.008 * price * promotion_spend
    + rng.normal(0.0, 8.0, sample_count)
)
data = pd.DataFrame({
    'price': price,
    'promotion_spend': promotion_spend,
    'operating_cost': operating_cost,
    'sales': sales,
})
data.head()


## Split training and test rows

The shuffled test rows stay out of `fit`. The same random seed makes this demonstration repeatable.


In [ ]:
order = rng.permutation(len(data))
split = int(0.8 * len(order))
train = data.iloc[order[:split]]
test = data.iloc[order[split:]]
feature_columns = ['price', 'promotion_spend', 'operating_cost']
X_train = train[feature_columns]
y_train = train['sales']
X_test = test[feature_columns]
y_test = test['sales']
print(f'train rows: {len(train)}, test rows: {len(test)}')


## Fit, predict, and evaluate


In [ ]:
model = MELBA(learning_rate=0.5, epochs=1800, random_state=2026)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
results = model.evaluate(X_test, y_test)
print('metrics:', results)
print('first five predictions:', predictions[:5])
print(f'optimizer updates: {model.n_iter_}, final objective: {model.loss_:.6f}')


In [ ]:
import matplotlib.pyplot as plt

lower = min(float(y_test.min()), float(predictions.min()))
upper = max(float(y_test.max()), float(predictions.max()))
plt.figure(figsize=(6, 5))
plt.scatter(y_test, predictions, alpha=0.75, color='#226c63')
plt.plot([lower, upper], [lower, upper], color='#a34f32', linewidth=1.5)
plt.xlabel('Actual synthetic sales')
plt.ylabel('Predicted sales')
plt.title('MELBA predictions on held-out synthetic data')
plt.tight_layout()
plt.show()


## Reading the result

Use the printed test metrics and scatter plot to inspect how closely predictions follow the synthetic targets. The diagonal line marks exact predictions. These results describe only this generated dataset; they do not establish performance on real business data or a causal effect of price, promotion, or cost.
